# 16a — Boltzmann Machine and Restricted Boltzmann Machine

Corresponds to: Note 16 — Boltzmann Machine and Restricted Boltzmann Machine

**Libraries introduced in this notebook:** No new libraries — numpy only, introduced in Note 03.

**What this notebook demonstrates:**
1. Stochastic neuron sampling — effect of temperature on firing probability
2. Boltzmann distribution — energies and probabilities for all states of a small network
3. RBM forward pass — P(h|v) and P(v|h) using conditional independence
4. CD-1 training loop — one full update step verified against Note 16 Part 4 worked example
5. CD-1 on a small binary dataset — weight convergence over multiple epochs

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

---
## Part 1: Stochastic Neuron Sampling

A stochastic neuron fires with probability P(sᵢ=1) = σ(netᵢ / T).

We show how temperature T changes the firing probability for the same net input,
and sample from the distribution to confirm the empirical frequency matches the probability.

In [ ]:
net = 2.0   # fixed net input
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

print(f"Stochastic neuron: net input = {net}")
print(f"{'T':>6}  {'P(s=1)':>10}  {'Empirical (10000 samples)':>26}")
print("-" * 48)

rng = np.random.default_rng(42)
for T in temperatures:
    p = sigmoid(net / T)
    samples = (rng.random(10000) < p).mean()
    print(f"{T:>6.1f}  {p:>10.4f}  {samples:>26.4f}")

print()
print("T→0: probability approaches 1.0 (deterministic firing for positive net input)")
print("T→∞: probability approaches 0.5 (random, ignores net input)")

---
## Part 2: Boltzmann Distribution

We compute the energy and probability of every state for the small network from Note 16 Part 2:
2 visible units, 1 hidden unit, weights w₁₁=1.0, w₂₁=0.5, all biases=0, T=1.

Expected: P(1,1,1) = 0.3236 — the lowest-energy state is the most probable.

In [ ]:
W_bm = np.array([[1.0],   # w_{v1, h1}
                 [0.5]])  # w_{v2, h1}
a = np.array([0.0, 0.0])  # visible biases
b = np.array([0.0])       # hidden biases

def energy_bm(v, h, W, a, b):
    return -(v @ W @ h) - (a @ v) - (b @ h)

states = []
for v1 in [0, 1]:
    for v2 in [0, 1]:
        for h1 in [0, 1]:
            v = np.array([v1, v2], dtype=float)
            h = np.array([h1], dtype=float)
            e = energy_bm(v, h, W_bm, a, b)
            states.append((v1, v2, h1, e))

Z = sum(np.exp(-s[3]) for s in states)

print(f"{'State (v1,v2,h1)':>18}  {'Energy':>8}  {'e^(-E)':>10}  {'P(s)':>8}")
print("-" * 52)
for v1, v2, h1, e in sorted(states, key=lambda s: s[3]):
    prob = np.exp(-e) / Z
    print(f"  ({v1}, {v2}, {h1})           {e:>8.4f}  {np.exp(-e):>10.4f}  {prob:>8.4f}")

print(f"\nZ = {Z:.4f}")
print(f"Sum of probabilities = {sum(np.exp(-s[3])/Z for s in states):.6f}  (should be 1.0)")
print()
print("Expected from Note 16 Part 2:")
print("  P(1,1,1) = 0.3236  (lowest energy, most probable)")
print("  P(1,0,1) = 0.1963")
print("  P(0,1,1) = 0.1190")

---
## Part 3: RBM Forward Pass

The RBM's conditional independence property means:
- Given v, all hidden units are independent: P(h|v) = Π_j P(hⱼ|v)
- Given h, all visible units are independent: P(v|h) = Π_i P(vᵢ|h)

We implement both directions and verify the Exercise 16 Section C numbers:
W = [[0.6, 0.4], [0.3, 0.8]], v⁽⁰⁾ = (1, 0).

In [ ]:
W = np.array([[0.6, 0.4],
              [0.3, 0.8]])
a = np.array([0.0, 0.0])   # visible biases
b = np.array([0.0, 0.0])   # hidden biases

def sample_h_given_v(v, W, b):
    """P(hⱼ=1|v) = sigmoid(W.T @ v + b)  — all hidden units in parallel."""
    return sigmoid(W.T @ v + b)

def sample_v_given_h(h, W, a):
    """P(vᵢ=1|h) = sigmoid(W @ h + a)  — all visible units in parallel."""
    return sigmoid(W @ h + a)

v0 = np.array([1.0, 0.0])

ph_given_v0 = sample_h_given_v(v0, W, b)
print(f"P(h|v⁽⁰⁾=(1,0)):")
print(f"  P(h1=1|v0) = {ph_given_v0[0]:.4f}  (expected 0.6457)")
print(f"  P(h2=1|v0) = {ph_given_v0[1]:.4f}  (expected 0.5987)")

h0 = ph_given_v0   # use expectations, not samples
pv_given_h0 = sample_v_given_h(h0, W, a)
print(f"\nP(v|h⁽⁰⁾):")
print(f"  P(v1=1|h0) = {pv_given_h0[0]:.4f}  (expected 0.6518)")
print(f"  P(v2=1|h0) = {pv_given_h0[1]:.4f}  (expected 0.6621)")

v1 = pv_given_h0
ph_given_v1 = sample_h_given_v(v1, W, b)
print(f"\nP(h|v⁽¹⁾):")
print(f"  P(h1=1|v1) = {ph_given_v1[0]:.4f}  (expected 0.6433)")
print(f"  P(h2=1|v1) = {ph_given_v1[1]:.4f}  (expected 0.6879)")

---
## Part 4: CD-1 Update Step

One full CD-1 update using the same network and training example.
Verified against Note 16 Part 4 worked example: w₁₁=0.5, w₂₁=0.3, v=(1,0).

In [ ]:
# Note 16 Part 4 worked example
W_ex = np.array([[0.5],
                 [0.3]])
a_ex = np.array([0.0, 0.0])
b_ex = np.array([0.0])
eta  = 0.1
v_ex = np.array([1.0, 0.0])

# Positive phase
ph0 = sample_h_given_v(v_ex, W_ex, b_ex)
print(f"Positive phase:")
print(f"  P(h1=1|v) = {ph0[0]:.4f}  (expected 0.6225)")

# Negative phase — one Gibbs step
pv1 = sample_v_given_h(ph0, W_ex, a_ex)
ph1 = sample_h_given_v(pv1, W_ex, b_ex)
print(f"\nNegative phase:")
print(f"  P(v1=1|h0) = {pv1[0]:.4f}  (expected 0.5771)")
print(f"  P(v2=1|h0) = {pv1[1]:.4f}  (expected 0.5466)")
print(f"  P(h1=1|v1) = {ph1[0]:.4f}  (expected 0.6113)")

# Weight update
pos = np.outer(v_ex, ph0)
neg = np.outer(pv1,  ph1)
dW  = eta * (pos - neg)
W_new = W_ex + dW

print(f"\nWeight updates:")
print(f"  dw11 = {dW[0,0]:.4f}  (expected  0.0270)")
print(f"  dw21 = {dW[1,0]:.4f}  (expected -0.0334)")
print(f"  w11_new = {W_new[0,0]:.4f}  (expected 0.5270)")
print(f"  w21_new = {W_new[1,0]:.4f}  (expected 0.2666)")

---
## Part 5: CD-1 Training on a Small Binary Dataset

We train a small RBM (4 visible, 3 hidden) on a binary dataset with two patterns:
- Pattern A: (1, 1, 0, 0) — first two units active
- Pattern B: (0, 0, 1, 1) — last two units active

After training, the RBM should assign high probability to reconstructions of these patterns.
We track reconstruction error over epochs to confirm convergence.

In [ ]:
def cd1_update(v, W, a, b, eta):
    """One CD-1 update. Returns updated W, a, b."""
    # Positive phase
    ph0 = sigmoid(W.T @ v + b)

    # Negative phase — one Gibbs step using expectations
    pv1 = sigmoid(W @ ph0 + a)
    ph1 = sigmoid(W.T @ pv1 + b)

    # Updates
    W_new = W + eta * (np.outer(v, ph0) - np.outer(pv1, ph1))
    a_new = a + eta * (v   - pv1)
    b_new = b + eta * (ph0 - ph1)
    return W_new, a_new, b_new, pv1

rng = np.random.default_rng(0)
n_vis, n_hid = 4, 3
W = rng.normal(0, 0.1, (n_vis, n_hid))
a = np.zeros(n_vis)
b = np.zeros(n_hid)
eta = 0.1

data = np.array([[1., 1., 0., 0.],
                 [0., 0., 1., 1.]])

print(f"{'Epoch':>6}  {'Recon error (A)':>16}  {'Recon error (B)':>16}")
print("-" * 44)

for epoch in range(1, 501):
    idx = rng.permutation(len(data))
    for i in idx:
        W, a, b, _ = cd1_update(data[i], W, a, b, eta)

    if epoch in [1, 10, 50, 100, 200, 500]:
        errors = []
        for v in data:
            ph = sigmoid(W.T @ v + b)
            pv_recon = sigmoid(W @ ph + a)
            errors.append(np.mean((v - pv_recon) ** 2))
        print(f"{epoch:>6}  {errors[0]:>16.4f}  {errors[1]:>16.4f}")

print()
print("Reconstruction error decreases toward zero — RBM has learned both patterns.")

Verify the trained RBM reconstructs each pattern correctly.

In [ ]:
print("Reconstruction quality after 500 epochs:")
for label, v in zip(['Pattern A (1,1,0,0)', 'Pattern B (0,0,1,1)'], data):
    ph = sigmoid(W.T @ v + b)
    pv_recon = sigmoid(W @ ph + a)
    print(f"  {label}:")
    print(f"    Input:         {v}")
    print(f"    Reconstruction: {pv_recon.round(3)}")

---
## Summary

| What was demonstrated | Key result |
|---|---|
| Stochastic neuron | P(s=1) = σ(net/T) — T→0 deterministic, T→∞ uniform random |
| Boltzmann distribution | P(1,1,1) = 0.3236 — lowest energy state is most probable; Z normalises all states to sum to 1 |
| RBM forward pass | P(h\|v) and P(v\|h) computed in parallel — conditional independence makes both exact |
| CD-1 update | Verified against Note 16 Part 4: dw11=0.0270, dw21=−0.0334 |
| CD-1 training | Reconstruction error converges to near zero on both patterns — RBM learns the data distribution |